In [4]:
import json
import os
import git
from util import *
from cal_statistics_util import *

In [5]:
valid_github_ds = []
with open(
    os.path.join(DATASET_DIR,"DS_GITHUB_EXTEND.json")
) as f:
    valid_github_ds = json.load(f)
print(len(valid_github_ds))

148


In [7]:
test_cids = set()
for info in valid_github_ds:
    info["repo_name"] = info["repo_name"].split("/")[1]
    info["bug_inducing_commit"] = info["bug_commit_hash"]
    test_cids.add(info["bug_fixing_commit"])

In [8]:
print("----DS_GITHUB-extended----")
precision, recall, f1_score,_ = get_metrics(valid_github_ds, test_cids)
print(
    f"llm4szz precision:{precision},llm4szz recall:{recall},llm4szz f1-score:{f1_score}"
)

----DS_GITHUB-extended----
llm4szz precision:0.6459497527057186,llm4szz recall:0.6036036036036037,llm4szz f1-score:0.6239695804357687


In [6]:
valid_linux_ds = []
with open(
    os.path.join(DATASET_DIR,'DS_LINUX_EXTEND.json')
) as f:
    valid_linux_ds = json.load(f)
print(len(valid_linux_ds))

500


In [10]:
test_cids = set()
for info in valid_linux_ds:
    info["bug_inducing_commit"] = info["bug_commit_hash"]
    test_cids.add(info["bug_fixing_commit"])

In [11]:
print("----DS_LINUX-extended----")
precision, recall, f1_score, _ = get_metrics(valid_linux_ds, test_cids)
print(
    f"llm4szz precision:{precision},llm4szz recall:{recall},llm4szz f1-score:{f1_score}"
)

----DS_LINUX-extended----
llm4szz precision:0.6420230943409752,llm4szz recall:0.5786666666666666,llm4szz f1-score:0.6086835133611816


In [12]:
def get_scalability_statistics(all_info, test_cids):
    all_token_costs = []
    all_call_llm_times = []
    all_elapsed_time = []
    for repeat_cnt in range(0, 3):
        token_costs = 0
        call_llm_times = 0
        elapsed_time = 0
        for info in all_info:
            repo_name = info["repo_name"]
            bug_fixing_cid = info["bug_fixing_commit"]
            if bug_fixing_cid not in test_cids:
                continue

            save_path = os.path.join(
                SAVE_LOG_DIR, repo_name, bug_fixing_cid, f"llm4szz{repeat_cnt}.json"
            )

            if os.path.exists(save_path):
                logs = []
                with open(save_path) as f:
                    logs = json.load(f)

                for log in logs:
                    if "token_cost" in str(log):
                        token_costs = token_costs + log["token_cost"]

                    if "call_llm_times" in str(log):
                        call_llm_times = call_llm_times + log["call_llm_times"]

                    if "elapsed_time" in str(log):
                        elapsed_time = elapsed_time + log["elapsed_time"]

        all_token_costs.append(token_costs / len(test_cids))
        all_call_llm_times.append(call_llm_times / len(test_cids))
        all_elapsed_time.append(elapsed_time / len(test_cids))

    return (
        sum(all_token_costs) / len(all_token_costs),
        sum(all_call_llm_times) / len(all_call_llm_times),
        sum(all_elapsed_time) / len(all_elapsed_time),
    )

In [13]:
all_info = []
with open(os.path.join(DATASET_DIR, "final_all_dataset.json")) as f:
    all_info = json.load(f)

test_info = []
with open(os.path.join(DATASET_DIR, "DS_LINUX.json")) as f:
    test_info = json.load(f)
test_cids = set()
for info in test_info:
    test_cids.add(info["fix_commit_hash"])

In [14]:
print('----DS_LINUX statistics----')
avg_token_costs, avg_call_llm_times, avg_elapsed_time = get_scalability_statistics(all_info,test_cids)
print(f'llm calls:{avg_call_llm_times},token numbers:{avg_token_costs},time:{avg_elapsed_time}')

----DS_LINUX statistics----
llm calls:9.769555555555556,token numbers:14489.235999999999,time:30.429324217478435


In [15]:
test_info = []
with open(os.path.join(DATASET_DIR, "DS_GITHUB.json")) as f:
    test_info = json.load(f)
test_cids = set()
for info in test_info:
    test_cids.add(info["fix_commit_hash"])

In [16]:
print('----DS_GITHUB statistics----')
avg_token_costs, avg_call_llm_times, avg_elapsed_time = get_scalability_statistics(all_info,test_cids)
print(f'llm calls:{avg_call_llm_times},token numbers:{avg_token_costs},time:{avg_elapsed_time}')

----DS_GITHUB statistics----


llm calls:10.282548476454293,token numbers:14890.397968605726,time:20.19769516479936


In [17]:
test_info = []
with open(os.path.join(DATASET_DIR, "DS_APACHE.json")) as f:
    test_info = json.load(f)
test_cids = set()
for info in test_info:
    test_cids.add(info["bug_fixing_commit"])

In [18]:
print('----DS_APACHE statistics----')
avg_token_costs, avg_call_llm_times, avg_elapsed_time = get_scalability_statistics(all_info,test_cids)
print(f'llm calls:{avg_call_llm_times},token numbers:{avg_token_costs},time:{avg_elapsed_time}')

----DS_APACHE statistics----


llm calls:13.665283540802212,token numbers:24023.970954356846,time:28.137354976738475
